# 3 1 count analysis he features

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import numpy as np
import pandas as pd
from utils import get_source_corrections, StataDiD, fit_columns

source_corrections = get_source_corrections(window_days=28).query('prior_corrections==0').reset_index(drop=True)
source_corrections = source_corrections.dropna(subset=fit_columns).reset_index(drop=True)
print(source_corrections.shape)
source_corrections.head()

In [ ]:
from utils import he_factors
from scipy.stats import zscore

for (col, col_name, col_type) in he_factors:
    if col_type == 'median':
        source_corrections[f'{col}_med'] = (source_corrections[col]>=source_corrections[col].median()).astype(int)
        source_corrections[f'{col}_q75'] = (source_corrections[col]>=source_corrections[col].quantile(0.75)).astype(int)
        if col == 'UserPostMisleadingness':
            print(f'Median: {source_corrections[col].median():.3f}, Q75: {source_corrections[col].quantile(0.75):.3f}')
        if col in ['user_follower_count', 'user_following_count']:
            source_corrections[col] = np.log1p(source_corrections[col])
        source_corrections[col] = zscore(source_corrections[col])
    elif col_type == 'alignment':
        source_corrections[col] = (source_corrections[col]>0).astype(int)
    else:
        source_corrections[col] = source_corrections[col].astype(int)

In [ ]:
from pystata import stata

stata.pdataframe_to_data(source_corrections.sort_values(by='event_creation_order', ascending=True).drop_duplicates(subset=['user_id']), force=True)

In [ ]:
from utils import get_coefs
fit_columns_mis = [col for col in fit_columns if col!='UserPostMisleadingness']
stata.run(f'logit UserPostMisleadingness_q75 {" ".join(fit_columns_mis)}, robust')
_, _, _ = get_coefs(reg='qui logit', start_line=0, end_line=len(fit_columns_mis), stat_type='z', irr_or=0, unit_group=False)

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
stata_did.inter_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', interv=fit_columns)
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=5, end_line=19, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
import matplotlib.pyplot as plt
from utils import FONT_SIZE, LW, MZ
plt.style.use('default')
plt.rcParams['font.size'] = FONT_SIZE+5

labels = ['Followers', 'Followees', 'Verified', 'Account\nage', 'Partisan', 'Misinfo\nexposure', 'Correction\ntiming', 'Frequency', 'Positive', 'Negative', 'Toxicity', 'Misleading', 'Confidence', 'Politics']
ylims = (-0.262, 0.462)

def coef_plot(means_mp, lls_mp, hls_mp, lbs, plot_name, grouplabel1='Account characteristics', grouplabel2='Correction timing and pre-correction posting preferences', y_group=0.36, ylim=None, selected_cols=1):
    # Sort by estimate (ascending)
    order = list(np.argsort(means_mp[:6])) + list(np.argsort(means_mp[6:])+6)

    estimate = np.array(means_mp)[order]
    ci_low = np.array(lls_mp)[order]
    ci_high = np.array(hls_mp)[order]
    lbs = np.array(lbs)[order]

    xticks = np.arange(len(estimate))

    plt.subplots(figsize=(16,6), layout='constrained')
    plt.axhline(y=0, linestyle='--', color='lightgrey', linewidth=LW)
    plt.axvline(x=5.5, linestyle='--', color='lightgrey', linewidth=LW)
    plt.text(-0.5, y_group, grouplabel1, size=FONT_SIZE-3)
    plt.text(5.7, y_group, grouplabel2, size=FONT_SIZE-3)
    plt.errorbar(x=xticks[:-selected_cols], y=estimate[:-selected_cols], yerr=[estimate[:-selected_cols]-ci_low[:-selected_cols], ci_high[:-selected_cols]-estimate[:-selected_cols]], marker='_', linestyle='', color='grey', markersize=MZ+2, mew=LW+2, linewidth=LW+2)
    plt.errorbar(x=xticks[-selected_cols:], y=estimate[-selected_cols:], yerr=[estimate[-selected_cols:]-ci_low[-selected_cols:], ci_high[-selected_cols:]-estimate[-selected_cols:]], marker='_', linestyle='', color='black', markersize=MZ+2, mew=LW+2, linewidth=LW+2)
    plt.xticks(xticks, lbs, rotation=60)
    if ylim is not None:
        plt.ylim(ylim)
    plt.ylabel('Interaction test')
    plt.savefig(f'outputs/figures/{plot_name}.pdf')
    plt.show()

coef_plot(means_mp_atts, lls_mp_atts, hls_mp_atts, labels, 'count_he_features', ylim=ylims, y_group=0.4)

In [ ]:
for (col, col_name, col_type) in he_factors:
    print('----------------------------')
    print(col_name)
    if col_type == 'median':
        col = f'{col}_med'
    print('Group 1')
    stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition=f'{col}==1')
    if col == 'UserPostMisleadingness_med':
        mean_agg_att_med_1, ll_agg_att_med_1, hl_agg_att_med_1 = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)
    else:
        _, _, _ = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)
    print('Group 0')
    stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition=f'{col}==0')
    if col == 'UserPostMisleadingness_med':
        mean_agg_att_med_0, ll_agg_att_med_0, hl_agg_att_med_0 = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)
    else:
        _, _, _ = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
for (col, col_name, col_type) in he_factors:
    print('----------------------------')
    print(col_name)
    if col_type == 'median':
        col = f'{col}_q75'
        print('Group 1')
        stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition=f'{col}==1')
        if col == 'UserPostMisleadingness_q75':
            mean_agg_att_q75_1, ll_agg_att_q75_1, hl_agg_att_q75_1 = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)
        else:
            _, _, _ = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)
        print('Group 0')
        stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition=f'{col}==0')
        if col == 'UserPostMisleadingness_q75':
            mean_agg_att_q75_0, ll_agg_att_q75_0, hl_agg_att_q75_0 = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)
        else:
            _, _, _ = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
import numpy as np
from utils import  vermilion
import math

plt.style.use('default')
plt.rcParams['font.size'] = FONT_SIZE
plt.subplots(layout='constrained')
estimates = [mean_agg_att_med_1, mean_agg_att_med_0, mean_agg_att_q75_1, mean_agg_att_q75_0]
lls = [ll_agg_att_med_1, ll_agg_att_med_0, ll_agg_att_q75_1, ll_agg_att_q75_0]
hls = [hl_agg_att_med_1, hl_agg_att_med_0, hl_agg_att_q75_1, hl_agg_att_q75_0]
xticks = ['Median', '75th percentile']
plt.axhline(y=0, linestyle='--', color='lightgrey', linewidth=LW)
for i in range(len(estimates)):
    xtick = math.floor(i/2)-0.15 if i/2==0 or i/2==1 else math.floor(i/2)+0.15
    marker = 's' if i/2==0 or i/2==1 else 'D'
    if i == 0:
        label='Higher'
    elif i == 1:
        label='Lower'
    else:
        label = ''
    plt.errorbar(xtick, estimates[i][0], yerr=[[estimates[i][0]-lls[i][0]], [hls[i][0]-estimates[i][0]]],
                 marker=marker, linestyle='', color=vermilion, markersize=MZ-5, mew=LW, linewidth=LW, label=label)
plt.xticks(np.arange(len(xticks)), xticks)
plt.xlim(-0.5, len(xticks)-0.5)
plt.ylim(ylims)
plt.xlabel('Group separation')
plt.ylabel('Estimated DiD effect')
plt.legend(title='Pre-correction misleadingness', ncol=2)
plt.savefig('outputs/figures/did_posts_subgroup_mis.pdf')
plt.show()

In [ ]:
from utils import SEED

stata.run(
    f"""
    sdid tweet_count unitId weeks_from_display treated_after if UserPostMisleadingness_med, ///
        vce(bootstrap) cluster(account_cluster) ///
        xline_opts(lcolor(none)) ///
        reps(100) seed({SEED}) graph ///
        g2_opt(xline(0, lcolor(black) lpattern(dash)) xtitle("Weeks from note display") ytitle("Post count") xlabel(-4(1)4) ylabel(25(5)45) title("") graphregion(color(white)))
    """
)

In [ ]:
stata.run(f"graph export outputs/figures/sdid_posts_weekly_mis_median.pdf, replace")

In [ ]:
stata.run(
    f"""
    sdid tweet_count unitId weeks_from_display treated_after if UserPostMisleadingness_q75, ///
        vce(bootstrap) cluster(account_cluster) ///
        xline_opts(lcolor(none)) ///
        reps(100) seed({SEED}) graph ///
        g2_opt(xline(0, lcolor(black) lpattern(dash)) xtitle("Weeks from note display") ytitle("Post count") xlabel(-4(1)4) ylabel(20(5)40) title("") graphregion(color(white)))
    """
)

In [ ]:
stata.run(f"graph export outputs/figures/sdid_posts_weekly_mis_q75.pdf, replace")

In [ ]:
stata.run('replace UserPostMisleadingness = UserPostMisleadingness_q75')

In [ ]:
stata_did.inter_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', interv=fit_columns)
_, _, _, pvals, _, _ = get_coefs(reg='qui ppmlhdfe', start_line=5, end_line=19, stat_type='z', irr_or=1, unit_group=True, full_return=True)

In [ ]:
stata.run(f'qui ppmlhdfe')
pvals = np.asarray(stata.get_return()['r(table)']).T.copy()[:,3][5:19]

In [ ]:
from statsmodels.stats.multitest import multipletests

def adjust_pvalues(p_values):
    return {
        method: multipletests(p_values, method=method)[1]
        for method in ('holm', 'fdr_bh')
    }

adjusted = adjust_pvalues(pvals)

print('Holm &   FDR-BH')
for holm, bh in zip(adjusted['holm'], adjusted['fdr_bh']):
    print(f'$p={holm:.3f}$  &   $p={bh:.3f}$')

### Replacement with prior NMR histories

In [ ]:
source_corrections = source_corrections.query('nmr_eligible_28d==1').reset_index(drop=True)
source_corrections['prior_nmr'] = (source_corrections['nmr_count_28d']>1).astype(int)
source_corrections = source_corrections.dropna(subset=fit_columns).reset_index(drop=True)
print(source_corrections.shape)
source_corrections.head()

In [ ]:
stata.pdataframe_to_data(source_corrections.query('treated==1'), force=True)
stata.run(f'logit prior_nmr {" ".join(fit_columns)}, robust')

In [ ]:
_, _, _ = get_coefs(reg='qui logit', start_line=0, end_line=len(fit_columns), stat_type='z', irr_or=0, unit_group=False)

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
stata_did.inter_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', interv=fit_columns+['prior_nmr'])
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=5, end_line=20, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
labels = ['Followers', 'Followees', 'Verified', 'Account\nage', 'Partisan', 'Misinfo\nexposure', 'Correction\ntiming', 'Frequency', 'Positive', 'Negative', 'Toxicity', 'Misleading', 'Confidence', 'Politics', 'prior NMR\n($\\geq 2$)']

coef_plot(means_mp_atts, lls_mp_atts, hls_mp_atts, labels, 'count_he_features_prior_nmr', grouplabel2='Correction timing, pre-correction posting preferences,\nand prior NMR-note history', ylim=ylims, y_group=0.35, selected_cols=2)